# Fase 4 (Parte A) — Modelos de Machine Learning Clásico
**Proyecto:** FruitVision — Clasificación de Calidad de Frutas  
**Curso:** Algoritmos y Programación III — Semestre 2026-1  
**Integrantes:** Juan José Gordillo Córdoba · Sebastián Jiménez Galvis · Juan Pablo Zambrano Cortez

---

### Arquitectura Híbrida FruitVision

La clasificación de **calidad** (Premium / Estándar / Descarte) es responsabilidad
exclusiva de los modelos de ML y la CNN.  
La estimación de **tamaño** (Pequeño / Mediano / Grande) es una función determinista
de geometría analítica (OpenCV) ya calculada en Fase 3.

### Modelos entrenados en este notebook

| Modelo | Descriptor | Hiperparámetros | Scoring CV |
|--------|-----------|----------------|-----------|
| Línea base | — | clase mayoritaria siempre | f1_macro |
| **Random Forest** | HSV-HS + LBP + Hu | `n_estimators`, `max_depth`, `min_samples_split` | f1_macro |
| **XGBoost** | HSV-HS + LBP + Hu | `learning_rate`, `max_depth`, `n_estimators` | f1_macro |

## 0. Configuración del entorno

In [ ]:
import sys
import pathlib
import time
import warnings
import joblib

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.feature import local_binary_pattern
from sklearn.dummy          import DummyClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics        import (accuracy_score, f1_score,
                                    classification_report,
                                    confusion_matrix,
                                    ConfusionMatrixDisplay)
from sklearn.preprocessing  import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ── Resolución robusta de la raíz del proyecto ────────────────────────────────
ROOT = pathlib.Path.cwd()
for _ in range(6):
    if (ROOT / "data").exists():
        break
    ROOT = ROOT.parent

DATA_PROC   = ROOT / "data" / "processed" / "fase3"
MODELS_DIR  = ROOT / "models" / "saved"
FIGURES_DIR = ROOT / "reports" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED             = 42
np.random.seed(SEED)
QUALITY_CLASSES  = ["Premium", "Estándar", "Descarte"]   # orden = id 0, 1, 2
QUALITY_PALETTE  = {"Premium": "#2ecc71", "Estándar": "#f39c12", "Descarte": "#e74c3c"}
N_CLASSES        = len(QUALITY_CLASSES)

print(f"Raíz del proyecto : {ROOT}")
print(f"Data procesada    : {DATA_PROC}")
print(f"Modelos           : {MODELS_DIR}")
print(f"Figuras           : {FIGURES_DIR}")
print(f"XGBoost           : {xgb.__version__}")

## 1. Carga de los manifests de Fase 3

In [ ]:
def load_manifest(split: str) -> pd.DataFrame:
    """
    Carga el manifest CSV generado por Fase 3.

    La columna 'processed_path' apunta a cada imagen PNG en disco.
    La columna 'quality_id' contiene el entero 0/1/2.
    """
    csv_path = DATA_PROC / f"manifest_{split}.csv"
    if not csv_path.exists():
        raise FileNotFoundError(
            f"No se encontró {csv_path}.\n"
            f"Asegúrate de haber ejecutado el notebook de Fase 3 correctamente."
        )
    df = pd.read_csv(csv_path)

    # Inferir quality_id desde 'quality' si no existe la columna directa
    if "quality_id" not in df.columns and "quality" in df.columns:
        q2id = {"Premium": 0, "Estándar": 1, "Descarte": 2}
        df["quality_id"] = df["quality"].map(q2id)

    # Verificar que processed_path existe para al menos la primera fila
    if len(df) > 0:
        sample_path = pathlib.Path(str(df["processed_path"].iloc[0]))
        if not sample_path.exists():
            print(f"[WARN] processed_path no encontrada: {sample_path}")
            print("       Intentando resolver relativa a ROOT...")
            df["processed_path"] = df["processed_path"].apply(
                lambda p: str((ROOT / pathlib.Path(p).relative_to(pathlib.Path(p).anchor))
                              .resolve()) if not pathlib.Path(p).exists() else p
            )

    n_valid   = df["quality_id"].notna().sum()
    n_invalid = df["quality_id"].isna().sum()
    print(f"[{split:5s}] {len(df):,} filas totales | {n_valid:,} con quality_id | {n_invalid} sin quality_id")
    if n_invalid > 0:
        print(f"  [WARN] Se omiten {n_invalid} filas sin quality_id en el split {split}.")

    return df[df["quality_id"].notna()].reset_index(drop=True)


manifest_train = load_manifest("train")
manifest_val   = load_manifest("val")
manifest_test  = load_manifest("test")

print(f"\nDistribución de calidad en cada split:")
for name, df_ in [("train", manifest_train), ("val", manifest_val), ("test", manifest_test)]:
    counts = df_["quality_id"].value_counts().sort_index()
    labels = {0: "Premium", 1: "Estándar", 2: "Descarte"}
    print(f"  {name:5s}: {' | '.join(f'{labels[k]}={v:,}' for k,v in counts.items())}  (total: {len(df_):,})")

## 2. Extracción de características — Descriptor HSV-HS + LBP + Momentos de Hu

El descriptor de cada imagen está formado por tres familias concatenadas:

$$\mathbf{d}(\mathcal{I}) = [\underbrace{h_H(\mathcal{I}),\, h_S(\mathcal{I})}_{\text{color HSV}},\; \underbrace{h_{\text{LBP}}(\mathcal{I})}_{\text{textura}},\; \underbrace{\boldsymbol{\eta}(\mathcal{I})}_{\text{forma}}]$$

| Familia | Dimensión | Qué captura |
|---------|-----------|-------------|
| Histograma H (Matiz) | 32 | Especie y madurez cromática |
| Histograma S (Saturación) | 32 | Viveza del color; zonas dañadas pierden saturación |
| LBP uniform (P=24, R=3) | 26 | Textura superficial: manchas, granulado, rugosidad |
| Momentos de Hu | 7 | Forma del contorno; invariante a rotación/escala |
| **Total** | **97** | — |

**Regla de oro del curso (fit/transform):**  
El `StandardScaler` hace `fit_transform` **solo en train**. En val y test, `transform` únicamente.
Los árboles (RF, XGBoost) no necesitan escalado pero se escala de todas formas para mantener
el pipeline uniforme y permitir comparaciones directas con otros clasificadores.

In [ ]:
# ── Constantes del extractor ─────────────────────────────────────────────────
N_BINS_H    = 32
N_BINS_S    = 32
LBP_RADIUS  = 3
LBP_NPOINTS = 8 * LBP_RADIUS      # 24
N_BINS_LBP  = LBP_NPOINTS + 2     # 26 (modo 'uniform')
N_HU        = 7                    # Momentos de Hu

FEAT_DIM    = N_BINS_H + N_BINS_S + N_BINS_LBP + N_HU
FEAT_NAMES  = (
    [f"H_bin_{i}"   for i in range(N_BINS_H)] +
    [f"S_bin_{i}"   for i in range(N_BINS_S)] +
    [f"LBP_bin_{i}" for i in range(N_BINS_LBP)] +
    [f"Hu_{i}"      for i in range(N_HU)]
)

# Slices de cada familia (para la gráfica de importancias)
SLICE_H   = slice(0, N_BINS_H)
SLICE_S   = slice(N_BINS_H, N_BINS_H + N_BINS_S)
SLICE_LBP = slice(N_BINS_H + N_BINS_S, N_BINS_H + N_BINS_S + N_BINS_LBP)
SLICE_HU  = slice(N_BINS_H + N_BINS_S + N_BINS_LBP, FEAT_DIM)

print(f"Dimensión del descriptor: {FEAT_DIM}")
print(f"  HSV-H : {N_BINS_H} dims  |  HSV-S : {N_BINS_S} dims")
print(f"  LBP   : {N_BINS_LBP} dims  |  Hu    : {N_HU} dims")


def extract_descriptor(img_path: str) -> np.ndarray:
    """
    Extrae el descriptor de 97 dimensiones de una imagen BGR en disco.

    Retorna un vector de ceros si la imagen no puede leerse (corrupción),
    marcando la fila con un flag que se filtrará antes del entrenamiento.

    --- Familia 1: Histograma HSV (H y S) ---
    Canal H ∈ [0, 180) en OpenCV (divide el rango 360° por 2).
    Canal S ∈ [0, 256).
    Cada histograma se normaliza dividiendo por su suma (estimación de densidad):
        p̂(b) = count(b) / Σ_b count(b)

    --- Familia 2: LBP (Local Binary Pattern) ---
    Para cada píxel central p_c, se comparan sus P vecinos sobre un círculo de radio R:
        LBP_{P,R}(x,y) = Σ_{i=0}^{P-1} s(p_i - p_c) · 2^i
    Modo 'uniform': solo patrones con ≤ 2 transiciones bit → P+2 bins.

    --- Familia 3: Momentos de Hu ---
    Invariantes a traslación, escala y rotación derivados de los momentos centrales.
    Se usa log10|η_i| para estabilizar la escala numérica (valores pueden diferir en órdenes de magnitud).
    """
    try:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            raise FileNotFoundError(str(img_path))
        img_bgr = cv2.resize(img_bgr, (128, 128), interpolation=cv2.INTER_AREA)

        # ── HSV H y S ─────────────────────────────────────────────────────────
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        h_hist = cv2.calcHist([hsv], [0], None, [N_BINS_H], [0, 180]).flatten()
        h_hist = h_hist / (h_hist.sum() + 1e-8)
        s_hist = cv2.calcHist([hsv], [1], None, [N_BINS_S], [0, 256]).flatten()
        s_hist = s_hist / (s_hist.sum() + 1e-8)

        # ── LBP ──────────────────────────────────────────────────────────────
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        lbp  = local_binary_pattern(gray, P=LBP_NPOINTS, R=LBP_RADIUS, method='uniform')
        lbp_hist, _ = np.histogram(lbp, bins=N_BINS_LBP,
                                   range=(0, LBP_NPOINTS + 2))
        lbp_hist = lbp_hist.astype(np.float32)
        lbp_hist = lbp_hist / (lbp_hist.sum() + 1e-8)

        # ── Momentos de Hu ────────────────────────────────────────────────────
        moments  = cv2.moments(gray)
        hu       = cv2.HuMoments(moments).flatten()
        # log10|η| para estabilizar escala numérica (algunos valores < 1e-10)
        hu_log   = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)

        return np.concatenate([h_hist, s_hist, lbp_hist, hu_log]).astype(np.float32)

    except (FileNotFoundError, cv2.error, Exception) as exc:
        print(f"[WARN] Fallo en {img_path}: {exc}")
        return np.zeros(FEAT_DIM, dtype=np.float32)


def build_feature_matrix(manifest: pd.DataFrame, split_name: str,
                         cache_dir: pathlib.Path = None) -> tuple:
    """
    Extrae o carga desde caché la matriz de características X y el vector de etiquetas y.

    FIX-ML-8: El scaler debe ajustarse (fit) SOLO en train. Esta función
    devuelve siempre X sin escalar; el escalado se aplica fuera, solo en train.

    Retorna (X, y) con filas válidas (excluye imágenes corruptas con desc. cero).
    """
    if cache_dir is None:
        cache_dir = DATA_PROC / "feature_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_X = cache_dir / f"X_{split_name}.npy"
    cache_y = cache_dir / f"y_{split_name}.npy"

    if cache_X.exists() and cache_y.exists():
        print(f"[{split_name:5s}] Cargando features desde caché...")
        X = np.load(cache_X)
        y = np.load(cache_y)
        print(f"  X={X.shape}  y={y.shape}")
        return X, y

    print(f"[{split_name:5s}] Extrayendo {FEAT_DIM}-dim features de {len(manifest):,} imágenes...")
    X_list, y_list = [], []
    for _, row in manifest.iterrows():
        feat = extract_descriptor(str(row["processed_path"]))
        X_list.append(feat)
        y_list.append(int(row["quality_id"]))

    X = np.vstack(X_list)
    y = np.array(y_list, dtype=np.int32)

    # Filtrar filas completamente en cero (imágenes corruptas)
    valid_mask = X.sum(axis=1) != 0
    n_corrupt  = (~valid_mask).sum()
    if n_corrupt > 0:
        print(f"  [WARN] {n_corrupt} imágenes corruptas filtradas.")
    X, y = X[valid_mask], y[valid_mask]

    np.save(cache_X, X)
    np.save(cache_y, y)
    print(f"  Guardado en caché: X={X.shape}  y={y.shape}")
    return X, y


# Extraer características (o cargar desde caché)
Xtr_raw, ytr = build_feature_matrix(manifest_train, "train")
Xva_raw, yva = build_feature_matrix(manifest_val,   "val")
Xte_raw, yte = build_feature_matrix(manifest_test,  "test")

# FIX-ML-8: Escalado — fit SOLO en train, transform en val y test
# (regla de oro del curso: "fit_transform en train, solo transform en test")
scaler = StandardScaler()
Xtr = scaler.fit_transform(Xtr_raw)
Xva = scaler.transform(Xva_raw)
Xte = scaler.transform(Xte_raw)

joblib.dump(scaler, MODELS_DIR / "feature_scaler.pkl")

print(f"\n✓ Matrices de características listas:")
print(f"  train : X={Xtr.shape}  y={ytr.shape}")
print(f"  val   : X={Xva.shape}  y={yva.shape}")
print(f"  test  : X={Xte.shape}  y={yte.shape}")
print(f"  Escalador guardado en models/saved/feature_scaler.pkl")

## 3. Línea base — Clasificador Mayoritario (DummyClassifier)

> **Regla de oro del curso:** el baseline es el clasificador más simple posible —
> predecir siempre la clase mayoritaria del conjunto de entrenamiento.  
> Todo modelo entrenado debe superar este umbral para justificar su complejidad.

In [ ]:
# FIX-ML-7: la baseline reporta f1_macro (misma métrica que GridSearchCV),
# no solo accuracy. Un modelo que predice siempre Premium tendrá
# accuracy ≈ proporción de Premium pero f1_macro ≈ 0 (las otras clases tienen 0 recall).

dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
dummy.fit(Xtr, ytr)
dummy_pred = dummy.predict(Xte)

base_acc = accuracy_score(yte, dummy_pred)
base_f1  = f1_score(yte, dummy_pred, average="macro")

print("═" * 60)
print("  Línea base — DummyClassifier (clase mayoritaria siempre)")
print("═" * 60)
print(f"  Accuracy   : {base_acc:.3f}")
print(f"  F1-macro   : {base_f1:.3f}   ← métrica principal")
print()
print(f"  Clase predicha siempre: {QUALITY_CLASSES[int(dummy.constant_[0])]}")
print()
print("  Todo modelo debe superar f1_macro > {:.3f} para ser útil.".format(base_f1))
print()
print(classification_report(yte, dummy_pred,
                             target_names=QUALITY_CLASSES, digits=3))

## 4. Random Forest + GridSearchCV

### Fundamento matemático

Random Forest es un ensemble de $B$ árboles de decisión $\{h_b(\mathbf{x})\}_{b=1}^B$, cada uno entrenado sobre una submuestra bootstrap del dataset y un subconjunto aleatorio $m = \lfloor\sqrt{d}\rfloor$ de las $d$ features:

$$\hat{y} = \underset{k}{\text{arg max}} \sum_{b=1}^{B} \mathbf{1}[h_b(\mathbf{x}) = k]$$

La aleatorización de features **decorrela** los árboles, reduciendo la varianza del ensemble sin aumentar el sesgo.

**Criterio de división (Gini):**
$$G(t) = 1 - \sum_{k=1}^{K} p_k^2$$

### ¿Por qué `class_weight='balanced'` para FruitVision?

Con IR ≈ 10.86, la clase Estándar es minoritaria. Sin penalización, el árbol de Gini minimiza el error global sacrificando el recall de Estándar. `class_weight='balanced'` escala el costo de error de clase $k$ por $n / (K \cdot n_k)$, compensando el desbalance.

### ¿Por qué `StratifiedKFold` en lugar de `KFold`?

El desbalance (IR ≈ 10.86) hace que la proporción de clases por fold pueda variar significativamente con `KFold` estándar, produciendo métricas optimistas o pesimistas según el fold. `StratifiedKFold` garantiza que cada fold tenga la misma proporción de clases que el dataset completo.

In [ ]:
# FIX-ML-2: usar StratifiedKFold explícitamente (no el KFold implícito de cv=5)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rf_param_grid = {
    "n_estimators"    : [200, 400],
    "max_depth"       : [None, 20, 30],
    "min_samples_split": [2, 5],
}

print("Iniciando GridSearchCV — Random Forest...")
print(f"  Grid: {rf_param_grid}")
print(f"  CV  : StratifiedKFold(n_splits=5, shuffle=True, random_state={SEED})")
print(f"  Scoring: f1_macro\n")

t0 = time.time()
rf_gs = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight="balanced",   # compensa el desbalanceo
        random_state=SEED,
        n_jobs=1,                  # paralelismo en el GridSearch externo
    ),
    param_grid=rf_param_grid,
    cv=cv_strategy,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1,
    refit=True,
)
rf_gs.fit(Xtr, ytr)

elapsed = time.time() - t0
print(f"\n[{elapsed:.0f} s] Búsqueda completada")
print(f"  Mejores hiperparámetros : {rf_gs.best_params_}")
print(f"  F1-macro CV (train)     : {rf_gs.best_score_:.3f}")

rf_val_f1 = f1_score(yva, rf_gs.predict(Xva), average="macro")
rf_val_acc = accuracy_score(yva, rf_gs.predict(Xva))
print(f"  F1-macro val            : {rf_val_f1:.3f}")
print(f"  Accuracy val            : {rf_val_acc:.3f}")

## 5. XGBoost + GridSearchCV

### Fundamento matemático

XGBoost minimiza un objetivo regularizado que combina la pérdida empírica con un término de regularización sobre la complejidad de cada árbol $f_t$:

$$\mathcal{J}(F) = \sum_{i=1}^n L(y_i, \hat{y}_i) + \sum_{t=1}^T \Omega(f_t), \quad \Omega(f_t) = \gamma T + \tfrac{1}{2}\lambda \|w\|^2$$

Cada árbol se ajusta sobre los **pseudo-residuos** (gradiente negativo de la pérdida), usando una aproximación de Taylor de segundo orden que incorpora el **Hessiano** $h_i = \partial^2 L / \partial \hat{y}_i^2$. Esto da a XGBoost una convergencia más precisa y estable que Gradient Boosting clásico.

**Para clasificación multiclase** se usa `multi:softprob` (probabilidades) con entropía cruzada categórica como pérdida. El parámetro `scale_pos_weight` solo existe para binario; el balance en multiclase se maneja vía `sample_weight` (calculado con `compute_class_weight`).

In [ ]:
# FIX-ML-4: calcular sample_weight para XGBoost multiclase
# compute_class_weight devuelve el peso w_k = n / (K * n_k)
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(ytr),
    y=ytr,
)
class_weight_dict = dict(enumerate(class_weights_arr))
sample_weight_tr  = np.array([class_weight_dict[y] for y in ytr])

print("Pesos de clase (XGBoost):")
for cid, cname in enumerate(QUALITY_CLASSES):
    n_k = (ytr == cid).sum()
    print(f"  {cname:10s} (n={n_k:,}) → w = {class_weight_dict[cid]:.4f}")

# FIX-ML-2: StratifiedKFold también para XGBoost
# FIX-ML-3: objective='multi:softprob', eval_metric='mlogloss'
xgb_param_grid = {
    "learning_rate" : [0.05, 0.1, 0.2],
    "max_depth"     : [4, 6],
    "n_estimators"  : [200, 400],
}

print("\nIniciando GridSearchCV — XGBoost...")
print(f"  Grid: {xgb_param_grid}")
print(f"  CV  : StratifiedKFold(n_splits=5, shuffle=True, random_state={SEED})")
print(f"  Scoring: f1_macro\n")

t0 = time.time()
xgb_gs = GridSearchCV(
    estimator=xgb.XGBClassifier(
        objective       = "multi:softprob",
        num_class       = N_CLASSES,
        eval_metric     = "mlogloss",
        use_label_encoder = False,   # FIX-ML-3: evita warning en XGBoost ≥ 1.6
        tree_method     = "hist",
        random_state    = SEED,
        n_jobs          = 1,
    ),
    param_grid = xgb_param_grid,
    cv         = cv_strategy,
    scoring    = "f1_macro",
    n_jobs     = -1,
    verbose    = 1,
    refit      = True,
)
# FIX-ML-4: pasar sample_weight al .fit() del pipeline interno
xgb_gs.fit(Xtr, ytr, **{"sample_weight": sample_weight_tr})

elapsed = time.time() - t0
print(f"\n[{elapsed:.0f} s] Búsqueda completada")
print(f"  Mejores hiperparámetros : {xgb_gs.best_params_}")
print(f"  F1-macro CV (train)     : {xgb_gs.best_score_:.3f}")

xgb_val_f1 = f1_score(yva, xgb_gs.predict(Xva), average="macro")
xgb_val_acc = accuracy_score(yva, xgb_gs.predict(Xva))
print(f"  F1-macro val            : {xgb_val_f1:.3f}")
print(f"  Accuracy val            : {xgb_val_acc:.3f}")

## 6. Análisis de resultados del GridSearch

In [ ]:
# Visualizamos el mapa de F1-macro del GridSearch para RF
cv_rf = pd.DataFrame(rf_gs.cv_results_)

pivot_n_depth = cv_rf.pivot_table(
    values  = "mean_test_score",
    index   = "param_max_depth",
    columns = "param_n_estimators",
    aggfunc = "max",
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análisis del GridSearchCV — F1-macro por combinación de hiperparámetros",
             fontsize=13, fontweight="bold")

# Heatmap RF
sns.heatmap(pivot_n_depth.astype(float), annot=True, fmt=".3f",
            cmap="YlOrRd", linewidths=0.5, ax=axes[0])
axes[0].set_title("Random Forest\n(F1-macro CV medio, maximizando sobre min_samples_split)")
axes[0].set_xlabel("n_estimators"); axes[0].set_ylabel("max_depth")

# Heatmap XGBoost
cv_xgb = pd.DataFrame(xgb_gs.cv_results_)
pivot_xgb = cv_xgb.pivot_table(
    values  = "mean_test_score",
    index   = "param_max_depth",
    columns = "param_n_estimators",
    aggfunc = "max",
)
sns.heatmap(pivot_xgb.astype(float), annot=True, fmt=".3f",
            cmap="YlOrRd", linewidths=0.5, ax=axes[1])
axes[1].set_title("XGBoost\n(F1-macro CV medio, maximizando sobre learning_rate)")
axes[1].set_xlabel("n_estimators"); axes[1].set_ylabel("max_depth")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fase4_ml_gridsearch.pdf", bbox_inches="tight")
plt.show()
print("[✓] Figura guardada → fase4_ml_gridsearch.pdf")

## 7. Evaluación en test — Métricas completas

In [ ]:
def evaluate_model(model, name: str, Xte: np.ndarray, yte: np.ndarray) -> dict:
    """
    Evalúa un modelo en el conjunto de test y reporta:
    - Accuracy, F1-macro, F1 por clase
    - Classification report
    - Matriz de confusión

    Retorna un dict con las métricas para la comparativa global.
    """
    pred     = model.predict(Xte)
    acc      = accuracy_score(yte, pred)
    f1_macro = f1_score(yte, pred, average="macro")
    f1_per   = f1_score(yte, pred, average=None, labels=[0, 1, 2])

    print(f"{'═'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    print(f"  Accuracy   : {acc:.3f}")
    print(f"  F1-macro   : {f1_macro:.3f}   ← métrica principal")
    print(f"  F1 Premium : {f1_per[0]:.3f}")
    print(f"  F1 Estándar: {f1_per[1]:.3f}   ← clase crítica (minoritaria)")
    print(f"  F1 Descarte: {f1_per[2]:.3f}")
    print()
    print(classification_report(yte, pred, target_names=QUALITY_CLASSES, digits=3))

    return {
        "modelo"   : name,
        "accuracy" : acc,
        "f1_macro" : f1_macro,
        "f1_Premium"  : f1_per[0],
        "f1_Estándar" : f1_per[1],
        "f1_Descarte" : f1_per[2],
        "pred"     : pred,
    }


results_list = [
    {"modelo": "Baseline (mayoritario)", "accuracy": base_acc,
     "f1_macro": base_f1, "f1_Premium": 0.0, "f1_Estándar": 0.0,
     "f1_Descarte": 0.0, "pred": dummy_pred},
]

rf_res  = evaluate_model(rf_gs,  "Random Forest", Xte, yte)
xgb_res = evaluate_model(xgb_gs, "XGBoost",       Xte, yte)
results_list.extend([rf_res, xgb_res])

### 7.1 Matrices de confusión (test)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle("Matrices de Confusión — Conjunto de Test", fontsize=14, fontweight="bold")

for ax, res, cmap in zip(axes,
                          [rf_res, xgb_res],
                          ["Blues", "Oranges"]):
    cm = confusion_matrix(yte, res["pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=QUALITY_CLASSES)
    disp.plot(ax=ax, cmap=cmap, colorbar=False)
    ax.set_title(f"{res['modelo']}\nF1-macro={res['f1_macro']:.3f}  |  Acc={res['accuracy']:.3f}",
                 fontsize=11)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fase4_ml_confusion.pdf", bbox_inches="tight")
plt.show()
print("[✓] Figura guardada → fase4_ml_confusion.pdf")

## 8. Importancia de características (Random Forest)

> La importancia de Gini mide cuánto reduce en promedio la impureza del nodo
> cada feature, ponderada por el número de muestras que pasan por ese nodo.
> **No es** una medida causal: solo refleja la utilidad discriminativa en este dataset.

In [ ]:
imp = rf_gs.best_estimator_.feature_importances_

# FIX-ML-5: importancias por familia calculadas dinámicamente desde los slices
families = {
    "HSV-H (color matiz)"   : imp[SLICE_H].sum(),
    "HSV-S (color saturación)": imp[SLICE_S].sum(),
    "LBP (textura)"         : imp[SLICE_LBP].sum(),
    "Hu (forma)"            : imp[SLICE_HU].sum(),
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Importancia de Características — Random Forest", fontsize=13, fontweight="bold")

# Gráfica de barras por familia
axes[0].bar(list(families.keys()), list(families.values()),
            color=["#3498db", "#2ecc71", "#e67e22", "#9b59b6"],
            edgecolor="white", linewidth=1.5)
for i, v in enumerate(families.values()):
    axes[0].text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")
axes[0].set_title("Importancia agregada por familia de descriptores")
axes[0].set_ylabel("Gini Importance (suma)"); axes[0].tick_params(axis="x", rotation=20)

# Top-25 features individuales
top25_idx = np.argsort(imp)[-25:][::-1]
top25_names = [FEAT_NAMES[i] for i in top25_idx]
top25_vals  = imp[top25_idx]
color_map   = {
    "H_"  : "#3498db",
    "S_"  : "#2ecc71",
    "LBP_": "#e67e22",
    "Hu_" : "#9b59b6",
}
colors_bar = [next(c for prefix, c in color_map.items() if n.startswith(prefix))
              for n in top25_names]

axes[1].barh(range(25), top25_vals[::-1], color=colors_bar[::-1], edgecolor="white")
axes[1].set_yticks(range(25))
axes[1].set_yticklabels(top25_names[::-1], fontsize=7)
axes[1].set_xlabel("Gini Importance")
axes[1].set_title("Top-25 features individuales")

# Leyenda de colores
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=f) for f, c in {
    "HSV-H": "#3498db", "HSV-S": "#2ecc71",
    "LBP"  : "#e67e22", "Hu"   : "#9b59b6"}.items()]
axes[1].legend(handles=legend_elements, fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fase4_ml_importancia.pdf", bbox_inches="tight")
plt.show()
print("[✓] Figura guardada → fase4_ml_importancia.pdf")
print()
print("Importancia agregada por familia:")
for fam, val in families.items():
    print(f"  {fam:30s}: {val:.4f}  ({val*100:.1f}%)")

## 9. Comparativa de modelos y guardado

In [ ]:
results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "pred"} for r in results_list
]).set_index("modelo")

print("Resumen comparativo en el test set:")
print(results_df[["accuracy", "f1_macro", "f1_Premium", "f1_Estándar", "f1_Descarte"]].round(3).to_string())

# ── Gráfica de comparativa ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
x_pos   = np.arange(len(results_df))
metrics_to_plot = [("accuracy", "#3498db"), ("f1_macro", "#e74c3c"),
                   ("f1_Estándar", "#f39c12")]
width   = 0.25

for i, (metric, color) in enumerate(metrics_to_plot):
    bars = ax.bar(x_pos + i * width, results_df[metric].values,
                  width=width, color=color, alpha=0.85, label=metric, edgecolor="white")
    for bar, val in zip(bars, results_df[metric].values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

ax.set_xticks(x_pos + width)
ax.set_xticklabels(results_df.index, rotation=10, fontsize=9)
ax.set_ylim(0, 1.0); ax.set_ylabel("Score"); ax.legend(fontsize=9)
ax.set_title("Comparativa de modelos ML — Conjunto de Test", fontsize=12, fontweight="bold")
ax.axhline(base_f1, color="gray", ls="--", lw=1.2, label="Baseline F1-macro")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fase4_ml_comparativa.pdf", bbox_inches="tight")
plt.show()

# ── Guardado de modelos y métricas ────────────────────────────────────────────
best_name, best_model, best_f1 = max(
    [("random_forest", rf_gs.best_estimator_,  rf_res["f1_macro"]),
     ("xgboost",       xgb_gs.best_estimator_, xgb_res["f1_macro"])],
    key=lambda t: t[2],
)
joblib.dump(best_model,              MODELS_DIR / "best_quality_ml.pkl")
joblib.dump(rf_gs.best_estimator_,  MODELS_DIR / "random_forest.pkl")
joblib.dump(xgb_gs.best_estimator_, MODELS_DIR / "xgboost.pkl")

# FIX-ML-6: guardar métricas CSV para que Fase 5 pueda importarlas
results_df.reset_index().to_csv(MODELS_DIR / "ml_metrics.csv", index=False)

print(f"\n✓ Mejor modelo ML : {best_name}  (F1-macro = {best_f1:.3f})")
print(f"  Guardado en      : models/saved/best_quality_ml.pkl")
print(f"  Métricas CSV     : models/saved/ml_metrics.csv")

---
## Próximos pasos → Fase 4 (Parte B): CNN

El notebook `cnn.ipynb` entrenará una CNN desde cero (3 bloques Conv→BN→MaxPool + Dense + Dropout)
sobre las mismas imágenes 256×256 y comparará sus métricas con RF y XGBoost.  
La comparación global (incluyendo baseline, tamaño, y análisis de errores) ocurre en **Fase 5**.

➡️ **Siguiente:** `cnn.ipynb`